# Task 2 — Hindi Sentence Ordering (Kaggle)

**Input:** the punctuated samples from Task 1 (`outputs/punctuated/sample-*.txt`)
**Model:** `bert-base-multilingual-cased` (mBERT, 178M params). mBERT was pre-trained with **Next Sentence Prediction**, so it already has a head that answers "does B come right after A?". We measure it zero-shot, fine-tune it on Hindi (lower 8 layers + embeddings frozen to stay under 1.5 GB), and keep whichever ranks better on validation (ROC-AUC).
**Ordering:** beam search over the pair-score matrix (+ TF-IDF similarity, weight tuned on validation) + local refinement
**Mixed topics:** TF-IDF clustering; feature type and split threshold tuned on synthetic mixed documents
**Output:** ordered sentences + groups of 5 sentences (input for Task 3)

### Setup (Notebook settings, right panel)
1. **Accelerator → GPU T4 x2**, **Internet → On**
2. **Add Input → Your Work → Notebooks → Task1_Hindi_Punctuation** (the *saved version* with the 8000-doc run).
   Backup: upload `task1_outputs.zip` as a dataset instead.
3. Run the cells one by one (Shift + Enter).

In [ ]:
# ===== CELL 1: Config =====
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re, json, glob, random, itertools, collections, math, shutil, time, zipfile
import numpy as np
import torch

SEED            = 42
MODEL_NAME      = "bert-base-multilingual-cased"   # mBERT with pre-trained Next-Sentence head
FOLLOW_IDX      = 0        # BERT NSP head: class 0 = "B follows A", class 1 = "B is random"
FREEZE_LAYERS   = 8        # freeze embeddings + lower 8 of 12 layers (memory)
LR              = 3e-5
NUM_DOCS        = 6000     # Sangraha docs (80% train / 10% val / 10% test)
MIN_SENT        = 4        # docs need at least this many sentences
MAX_SENT_TRAIN  = 40       # max sentences used from one training doc
EVAL_MIN, EVAL_MAX = 5, 20 # sentences per evaluation document
EVAL_DOCS       = 300      # docs for validation and for test
MAX_TRAIN_PAIRS = 100000
SIDE_WORDS      = 40       # last 40 words of A + first 40 words of B go into the model
MAX_LEN         = 160
EPOCHS          = 1
BEAM            = 64
MIN_CLUSTER     = 4        # a topic cluster must have >= 4 sentences
GROUP_SIZE      = 5
START_TOKEN     = "<start>"

WORK      = "/kaggle/working"
MODEL_DIR = f"{WORK}/order_model"
OUT_DIR   = f"{WORK}/outputs"
for d in [MODEL_DIR, OUT_DIR, f"{OUT_DIR}/ordered", f"{OUT_DIR}/groups"]:
    os.makedirs(d, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device == "cuda" else "NO GPU - turn on GPU T4 x2")

In [ ]:
# ===== CELL 2: Load the punctuated samples from Task 1 =====
found = glob.glob("/kaggle/input/**/punctuated/sample-*.txt", recursive=True)

if not found:   # backup: task1_outputs.zip uploaded as a dataset / present in notebook output
    for z in glob.glob("/kaggle/input/**/task1_outputs.zip", recursive=True):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(f"{WORK}/task1_outputs")
    found = glob.glob(f"{WORK}/task1_outputs/**/sample-*.txt", recursive=True)

assert found, "Task 1 punctuated samples not found. Add the Task1 notebook (saved version) as input."

def sample_key(p):
    m = re.search(r"sample-(\d+)", os.path.basename(p))
    return int(m.group(1)) if m else 999

by_name = {}
for p in sorted(found):
    by_name.setdefault(os.path.basename(p), p)
SAMPLE_FILES = sorted(by_name.values(), key=sample_key)
for p in SAMPLE_FILES:
    print(os.path.basename(p), "->", p)

In [ ]:
# ===== CELL 3: Text utilities (same tokenisation as Task 1, so formats match exactly) =====
PUNCT_TO_LABEL = {"।": "END", "॥": "END", ".": "END", "!": "END",
                  "?": "QUESTION", ",": "COMMA", ";": "COMMA"}
PRIORITY  = {"O": 0, "COMMA": 1, "END": 2, "QUESTION": 3}
PUNCT_OUT = {"O": "", "END": "।", "COMMA": ",", "QUESTION": "?"}
TOKEN_RE  = re.compile(r"\d+(?:[.,]\d+)+|[^\s।॥.,?!;]+|[।॥.,?!;]")
STRIP_CHARS = "\"'“”‘’()[]{}<>-–—…:|*#/\\`~_"

def clean_word(w):
    return w.strip(STRIP_CHARS)

def text_to_words_labels(text):
    words, labels = [], []
    for tok in TOKEN_RE.findall(text):
        if tok in PUNCT_TO_LABEL:
            if words and PRIORITY[PUNCT_TO_LABEL[tok]] > PRIORITY[labels[-1]]:
                labels[-1] = PUNCT_TO_LABEL[tok]
        else:
            w = clean_word(tok)
            if w:
                words.append(w); labels.append("O")
    return words, labels

def doc_to_sentences(text):
    # Normalise any punctuated text into sentences in the Task 1 output format.
    words, labels = text_to_words_labels(text)
    sents, cur = [], []
    for w, l in zip(words, labels):
        cur.append(w + PUNCT_OUT[l])
        if l in ("END", "QUESTION"):
            sents.append(" ".join(cur)); cur = []
    if cur:
        sents.append(" ".join(cur) + "।")
    return sents

def tail(s):  return " ".join(s.split()[-SIDE_WORDS:])
def head(s):  return " ".join(s.split()[:SIDE_WORDS])

def window(sents, max_len, rng):
    if len(sents) <= max_len:
        return list(sents)
    st = rng.randint(0, len(sents) - max_len)
    return sents[st:st + max_len]

SAMPLES = {}
for p in SAMPLE_FILES:
    s = doc_to_sentences(open(p, encoding="utf-8").read())
    SAMPLES[os.path.basename(p)] = s
    print(f"{os.path.basename(p)}: {len(s)} sentences | first: {s[0][:80]}")

In [ ]:
# ===== CELL 4: Stream Sangraha Hindi and split documents 80/10/10 =====
from datasets import load_dataset

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    pass

CACHE = f"{WORK}/sangraha_docs.json"
if os.path.exists(CACHE):
    all_docs = json.load(open(CACHE, encoding="utf-8"))
    print("Loaded cached documents from", CACHE)
else:
    stream = load_dataset("ai4bharat/sangraha", data_dir="verified/hin", split="train", streaming=True)
    stream = stream.shuffle(seed=SEED, buffer_size=10_000)
    all_docs = []
    for ex in stream:
        t = (ex.get("text") or "").strip()
        if "।" not in t and "?" not in t:
            continue
        sents = doc_to_sentences(t)
        if len(sents) >= MIN_SENT:
            all_docs.append(sents)
        if len(all_docs) >= NUM_DOCS:
            break
    json.dump(all_docs, open(CACHE, "w", encoding="utf-8"), ensure_ascii=False)

rng = random.Random(SEED)
order = list(range(len(all_docs))); rng.shuffle(order)
n_tr, n_va = int(0.8 * len(order)), int(0.1 * len(order))
train_docs = [all_docs[i] for i in order[:n_tr]]
val_docs   = [all_docs[i] for i in order[n_tr:n_tr + n_va]]
test_docs  = [all_docs[i] for i in order[n_tr + n_va:]]

lens = [len(d) for d in all_docs]
print(f"Docs: {len(all_docs)} | train {len(train_docs)} | val {len(val_docs)} | test {len(test_docs)}")
print(f"Sentences per doc: median {int(np.median(lens))}, mean {np.mean(lens):.1f}")
print("Example sentences:", train_docs[0][:3])

In [ ]:
# ===== CELL 5: Build training pairs (A, B, label = 1 if B directly follows A) =====
def make_pairs(doc_list, rng):
    A, B, Y = [], [], []
    for sents in doc_list:
        s = window(sents, MAX_SENT_TRAIN, rng); n = len(s)
        # which sentence starts the text?
        A.append(START_TOKEN); B.append(head(s[0])); Y.append(1)
        A.append(START_TOKEN); B.append(head(s[rng.randrange(1, n)])); Y.append(0)
        for i in range(n - 1):
            A.append(tail(s[i]));     B.append(head(s[i + 1])); Y.append(1)   # correct next
            A.append(tail(s[i + 1])); B.append(head(s[i]));     Y.append(0)   # reversed
            others = [j for j in range(n) if j not in (i, i + 1)]
            if others:                                                         # same doc, not next
                A.append(tail(s[i])); B.append(head(s[rng.choice(others)])); Y.append(0)
    idx = list(range(len(Y))); rng.shuffle(idx)
    return [A[i] for i in idx], [B[i] for i in idx], [Y[i] for i in idx]

rng = random.Random(SEED)
trA, trB, trY = make_pairs(train_docs, rng)
vaA, vaB, vaY = make_pairs(val_docs, rng)
trA, trB, trY = trA[:MAX_TRAIN_PAIRS], trB[:MAX_TRAIN_PAIRS], trY[:MAX_TRAIN_PAIRS]
vaA, vaB, vaY = vaA[:8000], vaB[:8000], vaY[:8000]

print(f"Train pairs: {len(trY)} (positive {np.mean(trY):.1%}) | Val pairs: {len(vaY)}")
print("Example positive:", next((a, b) for a, b, y in zip(trA, trB, trY) if y == 1 and a != START_TOKEN))

In [ ]:
# ===== CELL 6: Tokenise the pairs =====
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tok(batch):
    return tokenizer(batch["a"], batch["b"], truncation=True, max_length=MAX_LEN)

to_nsp = lambda ys: [FOLLOW_IDX if y == 1 else 1 - FOLLOW_IDX for y in ys]   # our 1 = follows -> NSP class 0
train_ds = Dataset.from_dict({"a": trA, "b": trB, "labels": to_nsp(trY)}).map(tok, batched=True, remove_columns=["a", "b"])
val_ds   = Dataset.from_dict({"a": vaA, "b": vaB, "labels": to_nsp(vaY)}).map(tok, batched=True, remove_columns=["a", "b"])
lens = [len(x) for x in train_ds["input_ids"][:5000]]
print(train_ds, val_ds)
print(f"Token length: median {int(np.median(lens))}, 95th pct {int(np.percentile(lens, 95))}, max allowed {MAX_LEN}")

In [ ]:
# ===== CELL 7: mBERT next-sentence model: zero-shot vs fine-tuned (keep the better one) =====
from transformers import (BertForNextSentencePrediction, Trainer,
                          TrainingArguments, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def load_nsp():
    m = BertForNextSentencePrediction.from_pretrained(MODEL_NAME)
    for p in m.bert.embeddings.parameters():
        p.requires_grad = False
    for layer in m.bert.encoder.layer[:FREEZE_LAYERS]:
        for p in layer.parameters():
            p.requires_grad = False
    return m

model = load_nsp()
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M")

def compute_metrics(ev):
    logits, labels = ev
    logits = torch.tensor(logits).float()
    p_follow = torch.softmax(logits, -1)[:, FOLLOW_IDX].numpy()
    y = (labels == FOLLOW_IDX).astype(int)
    pred = (logits.argmax(-1).numpy() == FOLLOW_IDX).astype(int)
    return {"auc": roc_auc_score(y, p_follow), "accuracy": accuracy_score(y, pred), "f1": f1_score(y, pred)}

BATCH, ACCUM = 2, 8      # effective batch 16 with tiny activations
total_steps = EPOCHS * math.ceil(len(train_ds) / (BATCH * ACCUM))
args = TrainingArguments(
    output_dir=f"{WORK}/checkpoints",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    per_device_eval_batch_size=32,
    learning_rate=LR,
    optim="adafactor",       # far smaller optimizer memory than Adam
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_steps=int(0.1 * total_steps),
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=max(1, total_steps // 4),
    save_strategy="steps",
    save_steps=max(1, total_steps // 4),
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="auc",
    logging_steps=200,
    report_to="none",
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)

zero_shot = trainer.evaluate()
print("ZERO-SHOT mBERT (no training):", {k: round(v, 4) for k, v in zero_shot.items() if k in ("eval_auc", "eval_accuracy", "eval_f1")})

if torch.cuda.is_available():
    # HARD LIMIT: this process may never use more than 1.45 GB of GPU memory (it would crash instead)
    _total = torch.cuda.get_device_properties(0).total_memory
    torch.cuda.set_per_process_memory_fraction(min(1.0, 1.45 * 1024**3 / _total), 0)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    print(f"GPU memory hard-capped at 1.45 GB (of {_total/1024**3:.1f} GB)")
trainer.train()
train_peak_gb     = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
train_reserved_gb = torch.cuda.max_memory_reserved() / 1024**3 if torch.cuda.is_available() else 0.0
print(f"\nPeak GPU memory during TRAINING: {train_peak_gb:.3f} GB allocated ({train_reserved_gb:.3f} GB reserved)")
print("UNDER 1.5 GB" if train_reserved_gb < 1.5 else "Reserved is at/over 1.5 GB -> set BATCH, ACCUM = 2, 8 and rerun this cell")

fine_tuned = trainer.evaluate()
print("FINE-TUNED mBERT:", {k: round(v, 4) for k, v in fine_tuned.items() if k in ("eval_auc", "eval_accuracy", "eval_f1")})
# NOTE: "Training Loss" in the table can look large because it adds up the accumulated mini-batches; judge by Validation Loss and AUC.

if fine_tuned["eval_auc"] >= zero_shot["eval_auc"]:
    CHOSEN_MODEL = "fine-tuned"; model = trainer.model
else:
    CHOSEN_MODEL = "zero-shot";  model = load_nsp()
pair_eval = {"zero_shot": zero_shot, "fine_tuned": fine_tuned, "chosen": CHOSEN_MODEL}
model = model.eval().to(device)
print("Using:", CHOSEN_MODEL)

In [ ]:
# ===== CELL 8: Scoring functions (pair model + TF-IDF) =====
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from scipy.sparse import hstack

tfidf_fit_sents = [s for d in train_docs for s in d][:150_000]
tf_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=100_000, sublinear_tf=True).fit(tfidf_fit_sents)
tf_word = TfidfVectorizer(analyzer="word", token_pattern=r"[^\s।?,]+", max_features=50_000, sublinear_tf=True).fit(tfidf_fit_sents)

tf_rare = TfidfVectorizer(analyzer="word", token_pattern=r"[^\s।?,]+", min_df=2, max_df=0.01, sublinear_tf=True).fit(tfidf_fit_sents)

def tfidf(sents):
    return normalize(hstack([tf_char.transform(sents), tf_word.transform(sents)]).tocsr())

FEATURE_KINDS = ["char+word", "word", "rare-word"]   # "rare-word" ignores words found in >1% of sentences
def topic_feats(sents, kind):
    if kind == "char+word":
        return tfidf(sents)
    if kind == "word":
        return normalize(tf_word.transform(sents))
    return normalize(tf_rare.transform(sents))

@torch.no_grad()
def pair_logprob_matrix(sents, batch_size=64):
    # Returns start[j] = log P(sentence j opens the text) and M[i, j] = log P(j directly follows i).
    n = len(sents)
    A  = [START_TOKEN] + [tail(s) for s in sents]
    Bh = [head(s) for s in sents]
    rows, cols = [], []
    for i in range(n + 1):
        for j in range(n):
            if i - 1 != j:
                rows.append(i); cols.append(j)
    out = np.full((n + 1, n), -np.inf)
    for k in range(0, len(rows), batch_size):
        r, c = rows[k:k + batch_size], cols[k:k + batch_size]
        enc = tokenizer([A[x] for x in r], [Bh[y] for y in c], truncation=True,
                        max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(**enc).logits
        out[r, c] = torch.log_softmax(logits.float(), -1)[:, FOLLOW_IDX].cpu().numpy()
    return out[0], out[1:]

def tfidf_sim(sents):
    C = cosine_similarity(tfidf(sents))
    np.fill_diagonal(C, -np.inf)
    return C
print("TF-IDF features:", tfidf(["परीक्षण वाक्य।"]).shape[1])

In [ ]:
# ===== CELL 9: Ordering + clustering algorithms =====
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score

def path_score(path, start, M):
    p = np.asarray(path)
    return float(start[p[0]] + (M[p[:-1], p[1:]].sum() if len(p) > 1 else 0.0))

def beam_order(start, M, beam=BEAM):
    n = len(start)
    if n == 1:
        return [0]
    k = min(beam, n)
    first = np.argsort(-start, kind="stable")[:k]
    paths  = [[int(j)] for j in first]
    scores = start[first].astype(np.float64)
    used = np.zeros((k, n), bool); used[np.arange(k), first] = True
    for _ in range(n - 1):
        last = np.array([p[-1] for p in paths])
        cand = np.where(used, -np.inf, scores[:, None] + M[last])
        flat = cand.ravel()
        kk = int(min(beam, np.isfinite(flat).sum()))
        top = np.argpartition(-flat, kk - 1)[:kk]
        top = top[np.argsort(-flat[top], kind="stable")]
        bi, nj = np.divmod(top, n)
        paths  = [paths[b] + [int(j)] for b, j in zip(bi, nj)]
        scores = flat[top]
        used = used[bi].copy(); used[np.arange(len(bi)), nj] = True
    return paths[int(np.argmax(scores))]

def improve(path, start, M, max_iter=300):
    # Local search: move blocks of 1-3 sentences to a better position while the total score rises.
    best = list(path); bs = path_score(best, start, M); n = len(best)
    for _ in range(max_iter):
        found = False
        for L in (1, 2, 3):
            for i in range(0, n - L + 1):
                seg, rest = best[i:i + L], best[:i] + best[i + L:]
                for j in range(len(rest) + 1):
                    if j == i:
                        continue
                    cand = rest[:j] + seg + rest[j:]
                    s = path_score(cand, start, M)
                    if s > bs + 1e-9:
                        best, bs, found = cand, s, True
                        break
                if found: break
            if found: break
        if not found:
            break
    return best

def cluster_candidates(X, max_k=4, min_size=MIN_CLUSTER):
    n = X.shape[0]
    D = cosine_distances(X); np.fill_diagonal(D, 0.0)
    res = {}
    for k in range(2, max_k + 1):
        if n < k * min_size:
            break
        for linkage in ("average", "complete"):
            lab = AgglomerativeClustering(n_clusters=k, metric="precomputed", linkage=linkage).fit_predict(D)
            if np.bincount(lab).min() < min_size:
                continue
            sil = silhouette_score(D, lab, metric="precomputed")
            if k not in res or sil > res[k][0]:
                res[k] = (sil, lab)
    return res

def choose_clusters(res, n, thresh):
    if not res:
        return np.zeros(n, int)
    k, (sil, lab) = max(res.items(), key=lambda kv: kv[1][0])
    return lab if sil >= thresh else np.zeros(n, int)

def order_with_clusters(start, M, labels, use_improve=True):
    orders = []
    for c in np.unique(labels):
        g = np.where(labels == c)[0]
        st, sub = start[g], M[np.ix_(g, g)]
        o = beam_order(st, sub)
        if use_improve:
            o = improve(o, st, sub)
        orders.append([int(g[i]) for i in o])
    if len(orders) == 1:
        return orders[0], orders
    best = None
    for perm in itertools.permutations(range(len(orders))):
        s = start[orders[perm[0]][0]] + sum(M[orders[perm[a]][-1], orders[perm[a + 1]][0]]
                                            for a in range(len(perm) - 1))
        if best is None or s > best[0]:
            best = (s, perm)
    seq = [orders[p] for p in best[1]]
    return [i for o in seq for i in o], seq

# ---- metrics (pred = gold positions listed in predicted order) ----
def kendall_tau(pred):
    n = len(pred)
    if n < 2: return 1.0
    inv = sum(1 for a in range(n) for b in range(a + 1, n) if pred[a] > pred[b])
    return 1 - 4 * inv / (n * (n - 1))

def adjacency_acc(pred):
    pairs = set(zip(pred[:-1], pred[1:]))
    return sum((i, i + 1) in pairs for i in range(len(pred) - 1)) / max(1, len(pred) - 1)

def all_metrics(pred):
    return {"tau": kendall_tau(pred), "pmr": float(pred == sorted(pred)),
            "adjacency": adjacency_acc(pred), "first": float(pred[0] == 0)}

# sanity check on a perfect score matrix
_n = 8; _M = np.full((_n, _n), -5.0); _st = np.full(_n, -5.0); _st[3] = 0
_true = [3, 1, 7, 0, 5, 2, 6, 4]
for a, b in zip(_true[:-1], _true[1:]): _M[a, b] = 0
np.fill_diagonal(_M, -np.inf)
assert beam_order(_st, _M) == _true and improve(_true, _st, _M) == _true
print("Algorithm self-test passed")

In [ ]:
# ===== CELL 10: Build val/test evaluation sets and score them once =====
def build_eval(doc_list, n_docs, seed):
    r = random.Random(seed); cases = []
    for sents in doc_list:
        if len(sents) < EVAL_MIN:
            continue
        w = window(sents, r.randint(EVAL_MIN, EVAL_MAX), r)
        perm = list(range(len(w))); r.shuffle(perm)
        shuffled = [w[p] for p in perm]
        cases.append({"perm": perm, "sents": shuffled})
        if len(cases) >= n_docs:
            break
    return cases

def score_cases(cases):
    for c in cases:
        c["start"], c["M"] = pair_logprob_matrix(c["sents"])
        c["C"] = tfidf_sim(c["sents"])
    return cases

t0 = time.time()
val_cases  = score_cases(build_eval(val_docs,  EVAL_DOCS, SEED + 1))
test_cases = score_cases(build_eval(test_docs, EVAL_DOCS, SEED + 2))
print(f"Scored {len(val_cases)} val + {len(test_cases)} test docs in {time.time()-t0:.0f}s | "
      f"mean sentences/doc: {np.mean([len(c['sents']) for c in test_cases]):.1f}")

In [ ]:
# ===== CELL 11: Tune on VALIDATION (TF-IDF weight alpha + topic-split threshold) =====
def run_method(c, method, alpha=0.0, use_improve=False):
    n = len(c["sents"])
    if method == "random":
        o = list(range(n)); random.Random(n * 7919 + len(c["perm"])).shuffle(o)
    elif method == "tfidf":
        o = beam_order(np.zeros(n), c["C"])
    else:
        M = c["M"] + alpha * np.where(np.isfinite(c["C"]), c["C"], 0.0)
        o = beam_order(c["start"], M)
        if use_improve:
            o = improve(o, c["start"], M)
    return [c["perm"][i] for i in o]

def evaluate(cases, **kw):
    ms = [all_metrics(run_method(c, **kw)) for c in cases]
    return {k: round(float(np.mean([m[k] for m in ms])), 4) for k in ms[0]}

ALPHAS = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]
alpha_res = {a: evaluate(val_cases, method="bert", alpha=a) for a in ALPHAS}
for a, r in alpha_res.items():
    print(f"alpha={a:<5} val {r}")
BEST_ALPHA = max(ALPHAS, key=lambda a: (alpha_res[a]["tau"], -a))
print("Chosen alpha:", BEST_ALPHA)

# --- topic splitting: choose feature type + threshold on synthetic mixtures of 1, 2 or 3 documents ---
def make_mixtures(doc_list, n_cases, seed):
    r = random.Random(seed); cases = []
    pool = [d for d in doc_list if len(d) >= MIN_SENT]
    for _ in range(n_cases):
        k = r.choice([1, 1, 2, 2, 3])
        sents, gold = [], []
        for ci, d in enumerate(r.sample(pool, k)):
            w = window(d, r.randint(MIN_SENT, 15), r)
            sents += w; gold += [ci] * len(w)
        cases.append((sents, np.array(gold)))
    return cases

def cand_cache(mixes, kind):
    return [(cluster_candidates(topic_feats(s, kind)), g, len(s)) for s, g in mixes]

mix_val_raw  = make_mixtures(val_docs,  300, SEED + 3)
mix_test_raw = make_mixtures(test_docs, 300, SEED + 4)
THRESHOLDS = [round(float(t), 2) for t in np.arange(0.0, 0.62, 0.02)] + [1.01]   # 1.01 = never split

best = None
for kind in FEATURE_KINDS:
    cc = cand_cache(mix_val_raw, kind)
    ari = {t: float(np.mean([adjusted_rand_score(g, choose_clusters(res, n, t)) for res, g, n in cc])) for t in THRESHOLDS}
    t_best = max(THRESHOLDS, key=lambda t: (ari[t], t))
    print(f"{kind:<10} best threshold {t_best:<5} val ARI {ari[t_best]:.3f}   (never split: {ari[1.01]:.3f})")
    if best is None or ari[t_best] > best[2]:
        best = (kind, t_best, ari[t_best])
BEST_KIND, BEST_THRESH, _ = best
print(f"Chosen topic features: {BEST_KIND}, threshold {BEST_THRESH}")

In [ ]:
# ===== CELL 12: Final TEST report (never used for tuning) =====
results = {
    "random baseline":            evaluate(test_cases, method="random"),
    "TF-IDF only":                evaluate(test_cases, method="tfidf"),
    f"mBERT next-sentence ({CHOSEN_MODEL})": evaluate(test_cases, method="bert", alpha=0.0),
    f"mBERT + TF-IDF (a={BEST_ALPHA})": evaluate(test_cases, method="bert", alpha=BEST_ALPHA),
    f"FINAL (+ local refinement)": evaluate(test_cases, method="bert", alpha=BEST_ALPHA, use_improve=True),
}
lines = [f"{'Method':<36}{'Kendall tau':>12}{'Perfect match':>15}{'Adjacency':>11}{'First sent.':>12}"]
for name, r in results.items():
    lines.append(f"{name:<36}{r['tau']:>12.4f}{r['pmr']:>15.4f}{r['adjacency']:>11.4f}{r['first']:>12.4f}")

mt = cand_cache(mix_test_raw, BEST_KIND)
pred_k = [len(np.unique(choose_clusters(res, n, BEST_THRESH))) for res, g, n in mt]
true_k = [len(np.unique(g)) for res, g, n in mt]
clus = {"ARI": float(np.mean([adjusted_rand_score(g, choose_clusters(res, n, BEST_THRESH)) for res, g, n in mt])),
        "topic_count_accuracy": float(np.mean(np.array(pred_k) == np.array(true_k)))}
lines.append(f"\nTopic clustering ({BEST_KIND}) on {len(mt)} synthetic test mixtures: ARI {clus['ARI']:.4f}, "
             f"correct number of topics {clus['topic_count_accuracy']:.1%}")
report = "\n".join(lines)
print(report)
with open(f"{OUT_DIR}/eval_task2.txt", "w", encoding="utf-8") as f:
    f.write(f"Test docs: {len(test_cases)}, {EVAL_MIN}-{EVAL_MAX} sentences each, shuffled\n\n" + report)

In [ ]:
# ===== CELL 13: Order the 7 samples + make groups of 5 =====
# If you check a sample and know its topic count, force it here, e.g. {"sample-3.txt": 3}
CLUSTER_OVERRIDE = {}

def force_k(sents, k):
    D = cosine_distances(topic_feats(sents, BEST_KIND)); np.fill_diagonal(D, 0.0)
    return AgglomerativeClustering(n_clusters=k, metric="precomputed", linkage="average").fit_predict(D)

if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

all_groups, cluster_info = {}, {}
for name, sents in SAMPLES.items():
    n = len(sents)
    start, M = pair_logprob_matrix(sents)
    C = tfidf_sim(sents)
    M = M + BEST_ALPHA * np.where(np.isfinite(C), C, 0.0)
    if name in CLUSTER_OVERRIDE:
        labels = force_k(sents, CLUSTER_OVERRIDE[name])
    else:
        labels = choose_clusters(cluster_candidates(topic_feats(sents, BEST_KIND)), n, BEST_THRESH)
    order, parts = order_with_clusters(start, M, labels)
    assert sorted(order) == list(range(n)), "ordering lost a sentence!"

    ordered = [sents[i] for i in order]
    groups = [ordered[i:i + GROUP_SIZE] for i in range(0, n, GROUP_SIZE)]
    all_groups[name] = groups
    cluster_info[name] = {"sentences": n, "topics": len(parts), "topic_sizes": [len(p) for p in parts]}

    with open(f"{OUT_DIR}/ordered/{name}", "w", encoding="utf-8") as f:
        f.write("\n".join(ordered))
    with open(f"{OUT_DIR}/groups/{name}", "w", encoding="utf-8") as f:
        f.write("\n\n".join(f"### Group {gi+1}\n" + " ".join(g) for gi, g in enumerate(groups)))
    print(f"{name}: {n} sentences -> {len(parts)} topic(s) {cluster_info[name]['topic_sizes']} -> {len(groups)} groups")
    if len(parts) > 1:
        for ti, p in enumerate(parts):
            print(f"   topic {ti+1}: {sents[p[0]][:70]}")

infer_peak_gb = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
with open(f"{OUT_DIR}/all_groups.json", "w", encoding="utf-8") as f:
    json.dump(all_groups, f, ensure_ascii=False, indent=1)
print(f"\nPeak GPU memory during ORDERING: {infer_peak_gb:.3f} GB")

In [ ]:
# ===== CELL 14: Preview one sample =====
name = list(all_groups)[0]
for gi, g in enumerate(all_groups[name][:3]):
    print(f"--- {name} | Group {gi+1} ---")
    for s in g:
        print("  ", s)

In [ ]:
# ===== CELL 15: Save model (Model 2), reports and zips =====
import joblib
from IPython.display import FileLink, display

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
joblib.dump({"tf_char": tf_char, "tf_word": tf_word, "tf_rare": tf_rare}, f"{MODEL_DIR}/tfidf.joblib")
settings = {"model": MODEL_NAME, "chosen": CHOSEN_MODEL, "follow_idx": FOLLOW_IDX, "alpha": BEST_ALPHA, "topic_features": BEST_KIND, "split_threshold": BEST_THRESH, "beam": BEAM, "side_words": SIDE_WORDS,
            "max_len": MAX_LEN, "start_token": START_TOKEN, "group_size": GROUP_SIZE, "min_cluster": MIN_CLUSTER}
json.dump(settings, open(f"{MODEL_DIR}/ordering_settings.json", "w"), indent=2)

report_json = {
    "model": MODEL_NAME, "total_params_M": round(total/1e6, 1), "trainable_params_M": round(trainable/1e6, 1),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "train_peak_gpu_gb": round(train_peak_gb, 3), "train_reserved_gpu_gb": round(train_reserved_gb, 3),
    "ordering_peak_gpu_gb": round(infer_peak_gb, 3),
    "under_1_5_gb": bool(train_reserved_gb < 1.5 and infer_peak_gb < 1.5),
    "pair_val": {k: {m: round(v[m], 4) for m in ("eval_auc", "eval_accuracy", "eval_f1")} for k, v in pair_eval.items() if k != "chosen"},
    "test_results": results, "clustering_test": clus, "settings": settings,
    "samples": cluster_info, "docs": {"train": len(train_docs), "val": len(val_docs), "test": len(test_docs)},
    "train_pairs": len(trY), "seed": SEED,
}
json.dump(report_json, open(f"{OUT_DIR}/report_task2.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(json.dumps({k: v for k, v in report_json.items() if k not in ("test_results",)}, ensure_ascii=False, indent=2))

shutil.rmtree(f"{WORK}/checkpoints", ignore_errors=True)
shutil.make_archive(f"{WORK}/order_model", "zip", MODEL_DIR)
shutil.make_archive(f"{WORK}/task2_outputs", "zip", OUT_DIR)
os.chdir(WORK)
display(FileLink("task2_outputs.zip"))
display(FileLink("order_model.zip"))
print("Done!")